# Optional external GPU reproduction
The active experiment is the explicitly approved **Qwen2.5-1.5B local laptop study**. This unexecuted notebook is a reproduction reference and runs the model selected by the uploaded configuration. It does not start cloud compute from your laptop.

To reproduce the original 7B design on a suitable external GPU, restore both archived `config/profiles/qwen2.5-7b-*.yaml` files to the active model/training paths before bundling. A 16 GiB T4 or larger is recommended for that archived configuration. Keep separate run IDs and do not mix 7B and 1.5B results.

Create `results/campusqa-source.zip` with `scripts/bundle.ps1`, then upload it below. IndicQA remains test-only; the approved IndicSQuAD Hindi/Tamil training extension is acquired by the pipeline. Read `docs/REPRODUCIBILITY.md`. Colab GPU availability and billing are controlled by your account. No research results are pre-filled.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, zipfile, subprocess, sys, json, time
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only campusqa-source.zip'
project = Path('/content/campusqa')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for item in archive.infolist():
        assert (project/item.filename).resolve().is_relative_to(project.resolve()), 'Unsafe archive path'
    archive.extractall(project)
os.chdir(project)
os.environ['CAMPUSQA_ROOT'] = str(project)
os.environ['HF_HOME'] = str(project/'.cache/huggingface')
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
print('Source ready:', project)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','torch==2.7.1','--index-url','https://download.pytorch.org/whl/cu126'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[ml,test]'],check=True)
subprocess.run([sys.executable,'-m','backend.cli','audit','--check-torch'],check=True)
from backend.hardware import audit, readiness
checked = readiness(audit(check_torch=True),training=True)
assert checked['ready'], checked['blockers']

In [ ]:
def command(*args):
    completed = subprocess.run([sys.executable,'-m','backend.cli',*args],capture_output=True,text=True)
    if completed.returncode:
        print(completed.stderr)
        raise RuntimeError('Command failed: '+ ' '.join(args))
    return json.loads(completed.stdout)
command('acquire')
manifest = command('prepare')
expected = json.loads((project/'handoff/expected_benchmark_manifest.json').read_text())
assert manifest['version'] == expected['version'], 'Reconstructed dataset differs from local handoff'
assert manifest['split_checksums'] == expected['split_checksums'], 'Dataset checksum mismatch'
print('Reproduced exact dataset version:',manifest['version'])
command('resolve-model')
print(command('tokenizer-check'))

## Supply independent training records if Hindi/Tamil are missing
The following step does not use IndicQA test records. Upload an actual training-only JSONL matching the importer schema, including original source IDs, canonical document IDs, language, context, question, reference answer, translation status, `split: train`, and `original_split: train`. A single file can contain both Hindi and Tamil. License and provenance must be reviewed. The importer does not establish semantic non-contamination just from a declared split label.

In [ ]:
if manifest['training_languages_missing']:
    print('Missing training languages:',manifest['training_languages_missing'])
    training_upload = files.upload()
    assert len(training_upload) == 1, 'Upload one reviewed training JSONL'
    training_file = project/'provided-training.jsonl'
    training_file.write_bytes(next(iter(training_upload.values())))
    source_name = input('Actual training corpus name: ').strip()
    source_license = input('Reviewed license (e.g. cc-by-4.0): ').strip()
    source_url = input('Actual official HTTPS provenance URL: ').strip()
    print(command('import-training',str(training_file),'--source-name',source_name,'--license',source_license,'--source-url',source_url))
    manifest = command('prepare')
assert not manifest['training_languages_missing'], manifest['training_languages_missing']
print('Final training/evaluation dataset version:',manifest['version'])
print(manifest['splits'])

In [ ]:
from backend.db import Database
def wait_for_run(run_id):
    while True:
        run = Database().run(run_id)
        print(run_id,run['status'],run['completed'],'/',run['total'])
        if run['status'] not in {'queued','running'}:
            assert run['status'] == 'completed', run.get('error')
            return run
        time.sleep(15)
baseline = command('run-evaluation','--track','benchmark','--experiment','A')
wait_for_run(baseline['run_id'])
print('Baseline complete:',baseline['run_id'])

In [ ]:
# Checkpoint selection uses only training/validation; never optimize from baseline test scores.
print(command('preflight'))
training_result = command('train')
print('Actual adapter:',training_result['adapter_path'])
adapted = command('run-evaluation','--track','benchmark','--experiment','B')
wait_for_run(adapted['run_id'])
print('LoRA run:',adapted['run_id'])
print('Human accuracy/hallucination metrics remain pending until reviewed annotations exist.')

## Continue campus research and preserve artifacts
The benchmark phase now permits official campus collection. Follow the reviewed FAQ, retrieval validation, sealed campus test, ingestion and A/B/C/D commands in the reproducibility guide. Campus FAQs stay out of fine-tuning. The code does not invent university policies or human annotations.

Run the next cell to archive actual data, run logs, registry, adapters and configuration for download. This can be large; review Colab storage and download capacity first. Original base-model weights and dependency caches are excluded. Keep the model revision pinned when restoring on another machine.

In [ ]:
archive_path = Path('/content/campusqa-experiment-artifacts.zip')
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for directory in ['data','results','config','models']:
        for file in (project/directory).rglob('*'):
            if file.is_file() and file != archive_path:
                archive.write(file,file.relative_to(project))
print('Archive bytes:',archive_path.stat().st_size)
files.download(str(archive_path))